# Avance 4 · Regresión de la potencia en la fase 2

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este cuaderno es el compañero del [04_Models](04_Models.ipynb) para la fase 2. Allí la comparativa de seis modelos es de **clasificación a 10 µM** y queda como el piso; aquí se mide lo que la fase 2 predice ahora: la **potencia** (`pActividad`) de un compuesto sobre una diana, y la clase se deriva de ella al final ([2.9](../docs/decisiones.md#d2-9)).

Son seis modelos individuales (sin ensambles), con el mismo protocolo que el 04. Se entrenan con las mediciones exactas: el modelo con censura es el baseline del [03b](03b_Baseline_regresion_actividad.ipynb), y su evidencia es lo que decide qué viene en el Avance 5.


## 1. Preparación

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo por modelo: la columna de tiempos es comparable

import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"

from src.evaluation.regresion import concordancia_harrell, rmse_exactas

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

### Los datos

La unidad es el par compuesto-diana. Se usan los mismos pares de entrenamiento y validación que el 04 (la prueba reservada no se toca) y la diana entra como columnas de 0 y 1. Las filas son las del baseline del 03b: **ninguna se descarta por no decidir a 10 µM**, y salen solo las que no informan nada (sin ninguna cota, o con cotas que se contradicen). Los modelos se entrenan con las exactas; la validación incluye las censuradas, porque la concordancia las usa.

In [ ]:
pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d = pares[pares["particion"].ne("prueba")].merge(desc, on="smiles").reset_index(drop=True)
cols = [c for c in desc.columns if c != "smiles"]

inferior = d["limite_inferior"].to_numpy(float)
superior = d["limite_superior"].to_numpy(float)
util = ~(np.isneginf(inferior) & np.isposinf(superior)) & (inferior <= superior)
print(f"{len(d):,} pares de entrenamiento y validación · {(~util).sum():,} sin información y salen")

d = d[util].reset_index(drop=True)
inferior, superior = inferior[util], superior[util]
# La diana entra como columnas 0/1: sin ella el modelo no sabe contra qué se midió.
X = np.hstack([d[cols].to_numpy("float32"), pd.get_dummies(d["target_gene"]).to_numpy("float32")])
exacta = inferior == superior
y_cls = d["activo_10um"].to_numpy(float)       # la clase derivada a 10 µM; NaN si no decide
g = d["grupo_cv"].to_numpy()
procedencia = d["estrato"].to_numpy()          # solo_chembl, solo_npass, mixto
print(f"{len(d):,} pares · {exacta.sum():,} con potencia exacta ({exacta.mean():.1%}) · "
      f"{(~exacta).sum():,} solo con cotas")

## 2. Seis modelos de regresión

| Clasificación (notebook 04) | Regresión (aquí) |
|:--|:--|
| logística | `Ridge` (regresión lineal) |
| árbol | `DecisionTreeRegressor` |
| kNN | `KNeighborsRegressor` |
| Naive Bayes | `BayesianRidge` (regresión lineal bayesiana) |
| SVM-RBF | `SVR` |
| MLP | `MLPRegressor` |

Naive Bayes no tiene versión de regresión; `BayesianRidge` es la familia bayesiana que sí la tiene. `media` (predecir siempre el promedio de las exactas) va aparte como **referencia**: es el piso y no cuenta entre los seis, porque no es un algoritmo.

**Métrica principal, fijada antes de ver los resultados: la concordancia de Harrell.** Es la única que usa también las mediciones censuradas, y la tabla se ordena por ella. Las otras son el Spearman y el RMSE sobre las exactas (declarado así: no dicen nada de las censuradas), y el MCC y el AUROC de la clase derivada, cortando la potencia predicha en 5 (= 10 µM), para poder compararlos con el 04.

**El mínimo que hay que superar** viene del [03b](03b_Baseline_regresion_actividad.ipynb): el mejor de cada métrica entre sus baselines, concordancia 0.700, Spearman 0.531 y RMSE 1.156. Un modelo que no los supere no aporta sobre una recta.

La validación es la del 04: cinco pliegues agrupados por `grupo_cv`, semillas 0, 1 y 2, que también fijan el azar de cada modelo.

In [ ]:
MODELOS = {
    "ridge":          make_pipeline(MinMaxScaler(), Ridge()),
    "arbol":          DecisionTreeRegressor(min_samples_leaf=5),
    "knn":            make_pipeline(MinMaxScaler(), KNeighborsRegressor(n_neighbors=15)),
    "bayesian_ridge": make_pipeline(StandardScaler(), BayesianRidge()),
    "svr":            make_pipeline(MinMaxScaler(), SVR()),
    "mlp":            make_pipeline(MinMaxScaler(), MLPRegressor(hidden_layer_sizes=(64,), max_iter=600)),
}
REFERENCIA = {"media": DummyRegressor()}        # el piso: no aprende nada
SEMILLAS = (0, 1, 2)
CORTE = 5.0                                      # pActividad 5 = 10 µM, el mismo corte del 04
MINIMOS = {"concordancia": 0.700, "spearman": 0.531, "rmse": 1.156}     # del 03b


def medir(inf, sup, clase, pred):
    """Concordancia con todas las mediciones; Spearman y RMSE con las exactas; MCC y AUROC de la clase derivada."""
    exa = inf == sup
    pe = pred[exa]
    rho = np.nan if np.ptp(pe) == 0 else spearmanr(inf[exa], pe).statistic   # nan si predice una constante
    rmse, _ = rmse_exactas(inf, sup, pred)
    fila = {"concordancia": concordancia_harrell(inf, sup, pred), "spearman": rho, "rmse": rmse,
            "mcc": np.nan, "auroc": np.nan}
    decide = ~np.isnan(clase)
    if len(np.unique(clase[decide])) == 2:
        fila["mcc"] = matthews_corrcoef(clase[decide], pred[decide] >= CORTE)
        fila["auroc"] = roc_auc_score(clase[decide], pred[decide])
    return fila


filas = []
estratificar = np.where(np.isnan(y_cls), -1, y_cls).astype(int)
for semilla in SEMILLAS:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=semilla)
    for k, (ajuste, val) in enumerate(cv.split(X, estratificar, g), 1):
        ajuste_ex = ajuste[exacta[ajuste]]                       # se entrena solo con las exactas
        for nombre, modelo in {**MODELOS, **REFERENCIA}.items():
            m = clone(modelo)
            # La semilla fija también el azar del modelo (árbol, MLP), no solo el reparto.
            m.set_params(**{p: semilla for p in m.get_params() if p.endswith("random_state")})
            reloj = time.perf_counter()
            m.fit(X[ajuste_ex], inferior[ajuste_ex])
            seg = time.perf_counter() - reloj
            pred = m.predict(X[val])
            base = {"modelo": nombre, "semilla": semilla, "pliegue": k, "seg": seg}
            filas.append({**base, "estrato": "todos",
                          **medir(inferior[val], superior[val], y_cls[val], pred)})
            for e in np.unique(procedencia[val]):
                sel = procedencia[val] == e
                ys = y_cls[val][sel]
                ys = ys[~np.isnan(ys)]
                if min(ys.sum(), (1 - ys).sum()) >= 20:         # estrato con 20 casos de cada clase
                    filas.append({**base, "estrato": e,
                                  **medir(inferior[val][sel], superior[val][sel], y_cls[val][sel], pred[sel])})
res = pd.DataFrame(filas)
print(f"{len(res):,} filas: {res['modelo'].nunique()} modelos × {len(SEMILLAS)} semillas × 5 pliegues, por estrato")

## 3. Resultados

In [ ]:
def tabla(res, estrato="todos"):
    """Media por modelo, ordenada por la métrica principal. `desv_concordancia` es dispersión entre
    evaluaciones, no error estándar: las semillas comparten filas."""
    r = res[res["estrato"] == estrato]
    t = r.groupby("modelo")[["concordancia", "spearman", "rmse", "mcc", "auroc", "seg"]].mean()
    t.insert(1, "desv_concordancia", r.groupby("modelo")["concordancia"].std())
    t["supera_minimos"] = ((t["concordancia"] > MINIMOS["concordancia"]).astype(int)
                           + (t["spearman"] > MINIMOS["spearman"]).astype(int)
                           + (t["rmse"] < MINIMOS["rmse"]).astype(int))
    return t.sort_values("concordancia", ascending=False).round(3)


tabla_todos = tabla(res)
tabla_todos

### Por procedencia

Un promedio puede esconder cómo le va a los naturales. En el 03b la concordancia de las rectas sube de 0.53 a 0.64 solo en los pares que solo tiene NPASS, y el control del Avance 3 mostró que el modelo separa mucho peor esos pares que los de ChEMBL.

In [ ]:
por_procedencia = {e: tabla(res, e) for e in ("solo_chembl", "mixto", "solo_npass")
                   if (res["estrato"] == e).any()}
pd.concat(por_procedencia, names=["procedencia"])[["concordancia", "desv_concordancia", "spearman", "rmse", "mcc", "auroc"]]

## 3.1 Lectura

(se escribe después de ver los resultados)

In [ ]:
# Registro en experimentos/ (decisión 2.9: la potencia como variable principal).
from src.tracking import Corrida

with Corrida("comparativa-regresion-seis-modelos", decision="2.9",
             params={"modelos": list(MODELOS), "referencia": list(REFERENCIA), "pliegues": 5,
                     "entrena_con": "solo exactas", "objetivo": "pActividad",
                     "metrica_principal": "concordancia", "corte_mcc": CORTE,
                     "minimos_03b": MINIMOS, "estratos": "procedencia"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "fase2_pares.parquet",
                    PROCESADOS / "fase2_descriptores_limpios.parquet"]) as c:
    c.metricas({"todos": tabla_todos.to_dict(orient="index"),
                **{e: t.to_dict(orient="index") for e, t in por_procedencia.items()}})

print(c.carpeta.name)

# Declaración de uso de IA y contribución

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Revisar este cuaderno sobre la versión de Antonio: sexto modelo, métricas de regresión, el criterio de las filas, el mínimo del 03b, la corrida registrada y la lectura | Todas | **Lo que el equipo cambió:** `media` sale de los seis, porque no es un algoritmo, y entra `BayesianRidge`; las métricas pasan de MAE a concordancia, Spearman y RMSE sobre las exactas; las filas ya no se filtran por decidir a 10 µM; el kernel pasa a `python3` y se corrigen las erratas. La lectura de la sección 3.1 y la ejecución del cuaderno las revisan Ingrid Pamela Ruiz Puga y José Antonio Vázquez Martínez (por confirmar en la revisión del PR) |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | |
| Artemio Santiago Padilla Robles | |
| José Antonio Vázquez Martínez | |

Las salidas de la herramienta se usaron como insumo de trabajo y el equipo asume la responsabilidad por la exactitud, la integridad y el contenido final del cuaderno.
